# 17 · Chunking long text
Level L3 · Part 3 · builds on lesson 16

Needs: nothing. Everything runs in local mode (`QdrantClient(":memory:")`). One `needs-download`
cell embeds every chunk with MiniLM (about 90 MB, cached once, then about 20 s on a CPU); when it
is skipped, the saved vectors in `data/chunk_embeddings_minilm.npz` are used.

## Goal
By the end of this lesson you can split long recipe instructions into chunks, measure with
lesson 16's metrics which chunk size retrieves best, and return one result per recipe with
`query_points_groups`.

## The idea
So far each recipe was one short description and one vector. Real documents are long: a recipe's
method has ten steps, and the answer to "how long do I rest the dough?" is one sentence in the
middle. Squeezing the whole method into one vector blurs that sentence into everything else, and
the model may not even read the end. **Chunking** cuts each long text into smaller pieces,
**chunks**, and gives each its own vector. Each chunk remembers which recipe it came from, its
**parent document**, so results can be shown one recipe at a time. Think of a cookbook index:
it points to the page with the answer, not just to "the book".

## Picture

```mermaid
flowchart LR
    R["recipe method<br/>~150-320 words"] --> C["chunk_words(size, overlap)"]
    C --> P1["chunk 0"] & P2["chunk 1"] & P3["chunk 2 ..."]
    P1 & P2 & P3 --> COL["collection 'chunks'<br/>payload: slug, recipe_id, chunk, text"]
    Q["'how long do I rest the dough?'"] --> G["query_points_groups<br/>group_by='recipe_id'"]
    COL --> G
    G --> OUT["one group per recipe:<br/>best chunk(s) + parent from 'recipes'"]
```

## Step by step
### 1. Long recipe methods
The course data now has full methods for 24 of the original recipes:
`data/recipe_methods_24.json` maps a slug to its list of steps. Load them with the recipe table
from lesson 16, and join each method's steps into one text.

In [1]:
import json, math, os, uuid, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress")
from qdrant_client import QdrantClient, models

recipes = pd.read_csv("../data/recipes_2000.csv").set_index("slug")
methods = json.load(open("../data/recipe_methods_24.json"))
docs = {slug: " ".join(steps) for slug, steps in methods.items()}
words = pd.Series({slug: len(text.split()) for slug, text in docs.items()})
print(len(docs), "methods | words per method: min", words.min(), "median", int(words.median()),
      "max", words.max())
print("longest:", words.sort_values().tail(3).to_dict())

24 methods | words per method: min 95 median 153 max 324
longest: {'vegetable-samosas': 258, 'apple-pie': 286, 'margherita-pizza': 324}


The methods are 95 to 324 words, ten times longer than lesson 15's descriptions. Here is the
start of the longest one:

In [2]:
print(docs["margherita-pizza"][:300], "...")

Put 500 g strong white flour, 7 g fast-action yeast and 10 g salt in a large bowl and stir them together. Pour in 325 ml lukewarm water and 2 tablespoons olive oil, then mix with a wooden spoon until a shaggy dough forms and no dry flour is left. Tip the dough onto a lightly floured surface and knea ...


### 2. A chunker: windows of words
The simplest chunker slides a window over the words. `size` is the **chunk size** (words per
chunk). `overlap` is how many words each chunk repeats from the end of the previous one, so a
sentence cut at a boundary still appears whole in one of them. The window moves forward by
`size - overlap` words each time, and stops once the rest of the text is already covered.

In [3]:
def chunk_words(text, size, overlap=0):
    words = text.split()
    step = size - overlap
    return [" ".join(words[start:start + size])
            for start in range(0, max(len(words) - overlap, 1), step)]

tiny = "one two three four five six seven eight nine ten"
print(chunk_words(tiny, size=4))
print(chunk_words(tiny, size=4, overlap=2))

['one two three four', 'five six seven eight', 'nine ten']
['one two three four', 'three four five six', 'five six seven eight', 'seven eight nine ten']


Without overlap, ten words become three chunks of 4, 4 and 2 words. With `overlap=2`, each chunk
starts with the last two words of the one before, so there are four chunks. Now the pizza
method in chunks of 50 words:

In [4]:
pizza_chunks = chunk_words(docs["margherita-pizza"], size=50)
print(len(pizza_chunks), "chunks of", [len(c.split()) for c in pizza_chunks], "words")
print("chunk 1:", pizza_chunks[1])

7 chunks of [50, 50, 50, 50, 50, 50, 24] words
chunk 1: Tip the dough onto a lightly floured surface and knead for 10 minutes, pushing it away with the heel of your hand and folding it back, until it is smooth, springy and slightly tacky. Put the dough in an oiled bowl, cover with a damp tea towel and leave it


Seven chunks. Chunk 1 starts at a sentence but stops in the middle of one: "...leave it", and
"to rest ... for 1 to 2 hours" opens chunk 2. Word windows don't know where sentences end.
Step 11 measures how often that cuts an answer, and Try it chunks by whole steps instead.

### 3. Embed every chunk once
The lesson compares several chunkings: the whole method as one piece (`None`) and windows of
200, 100, 50, 25 and 12 words, plus three overlaps. `CONFIGS` lists them as `(size, overlap)`.
`chunks_for` returns one recipe's pieces for a configuration.

In [5]:
CONFIGS = [(None, 0), (200, 0), (100, 0), (50, 0), (25, 0), (12, 0), (50, 10), (50, 25), (25, 10)]

def chunks_for(slug, size, overlap=0):
    return [docs[slug]] if size is None else chunk_words(docs[slug], size, overlap)

all_texts = sorted({c for size, overlap in CONFIGS for slug in docs
                    for c in chunks_for(slug, size, overlap)})
print(len(all_texts), "distinct chunk texts across", len(CONFIGS), "configurations")

1040 distinct chunk texts across 9 configurations


The model also embeds the test questions (step 7), one pizza variant used in step 4, and every
single step (used in Try it). `QUERIES` is the question list, written out here so the cache
can be built before the labels.

In [6]:
QUERIES = [
    "how long do I rest the dough?", "how do I check it is cooked in the middle?",
    "what temperature should the oil be for frying?", "how long should the meat rest before carving?",
    "how do I keep the eggs from scrambling?", "how much water do I need for the rice?",
    "how long does it take to caramelise onions?", "how do I get crispy skin?",
    "how long does it need to set in the fridge?", "how much gelatine should I use?",
    "how long do I soak dried chickpeas?", "when do I add the basil?",
    "why let it rest before cutting?", "how do I freeze it without a machine?",
    "how long do I soak the noodles?", "how long should I simmer the broth?",
    "how do I stop a soggy bottom crust?", "when should I flip them?",
    "why chill the dough before baking?", "how do I make the curry paste fragrant?",
]
pizza_no_basil = " ".join(methods["margherita-pizza"][:-1])     # the method without its last step
extra_texts = [pizza_no_basil] + [s for steps in methods.values() for s in steps]
CACHE_FILE = "../data/chunk_embeddings_minilm.npz"
MODEL = "sentence-transformers/all-MiniLM-L6-v2"
model = None
print(len(QUERIES), "queries,", len(extra_texts), "extra texts")

20 queries, 187 extra texts


The model cell (skipped without `TUT_ENABLE=download`) embeds everything and writes the cache.
It also counts each method's tokens: the model reads at most `max_seq_length` of them.

In [7]:
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_VERBOSITY", "error")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL, device="cpu")
texts = all_texts + [t for t in extra_texts if t not in set(all_texts)]
tokens = {s: len(model.tokenizer(docs[s], verbose=False)["input_ids"]) for s in docs}
np.savez_compressed(CACHE_FILE, model=MODEL, max_seq_length=model.max_seq_length,
         texts=texts, emb=model.encode_document(texts, batch_size=64),
         queries=QUERIES, query_emb=model.encode_query(QUERIES),
         token_slugs=list(tokens), token_counts=list(tokens.values()))
print("saved", len(texts), "chunk vectors and", len(QUERIES), "query vectors")

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


`embed` and `embed_query` read the cache first and fall back to the model, the same pattern as
lesson 16.

In [8]:
cache = np.load(CACHE_FILE)
assert str(cache["model"]) == MODEL
text_cache = dict(zip(cache["texts"].tolist(), cache["emb"], strict=True))
query_cache = dict(zip(cache["queries"].tolist(), cache["query_emb"], strict=True))

def embed(texts):
    missing = [t for t in texts if t not in text_cache]
    if missing and model is None:
        raise KeyError(f"{len(missing)} texts are not in the cache and the model is not loaded")
    if missing:
        text_cache.update(zip(missing, model.encode_document(missing)))
    return [text_cache[t].tolist() for t in texts]

def embed_query(text):
    if text not in query_cache and model is not None:
        query_cache[text] = model.encode_query(text)
    return query_cache[text].tolist()

print("cached chunk vectors:", len(text_cache), "| query vectors:", len(query_cache))

cached chunk vectors: 1224 | query vectors: 20


### 4. Why one vector per recipe is not enough
An embedding model reads a limited number of **tokens** (word pieces). MiniLM's limit is 256;
anything after it is cut off without a warning. Count the methods that are longer:

In [9]:
MAX_TOKENS = int(cache["max_seq_length"])
token_counts = pd.Series(cache["token_counts"], index=cache["token_slugs"])
too_long = token_counts[token_counts > MAX_TOKENS].sort_values()
print("model reads at most", MAX_TOKENS, "tokens")
print(len(too_long), "of", len(token_counts), "methods are longer:", too_long.to_dict())

model reads at most 256 tokens
5 of 24 methods are longer: {'sushi-rolls': 274, 'lasagne': 292, 'vegetable-samosas': 321, 'apple-pie': 354, 'margherita-pizza': 387}


The pizza method is 387 tokens, so the model never sees its last 131. Proof: embed the
method with and without its last step ("Scatter fresh basil leaves ... after it comes out of the
oven"). If the model had read that step, the two vectors would differ.

In [10]:
whole, no_basil = np.array(embed([docs["margherita-pizza"], pizza_no_basil]))
print("cosine(whole pizza method, method without the basil step) =", round(float(whole @ no_basil), 4))

cosine(whole pizza method, method without the basil step) = 1.0


The cosine is 1.0: the two vectors are the same, so a whole-recipe vector cannot answer "when
do I add the basil?" from the pizza recipe. Even for short methods, one vector is an average
of ten steps; a question about one step has to compete with the other nine.

### 5. One point per chunk, with its parent
Store 50-word chunks. Each point carries its parent's `slug` and `recipe_id`, its position
(`chunk`) and its `text`, so a hit can be shown and traced back. Ids are deterministic
(lesson 14): `uuid5` of `slug#index`, so every chunk gets its own id.

In [11]:
RECIPE_NS = uuid.uuid5(uuid.NAMESPACE_URL, "https://example.com/recipes/")

def recipe_id(slug):
    return str(uuid.uuid5(RECIPE_NS, slug))

def chunk_points(size, overlap=0):
    points = []
    for slug in docs:
        pieces = chunks_for(slug, size, overlap)
        for i, (text, vector) in enumerate(zip(pieces, embed(pieces))):
            points.append(models.PointStruct(
                id=str(uuid.uuid5(RECIPE_NS, f"{slug}#{i}")), vector=vector,
                payload={"slug": slug, "recipe_id": recipe_id(slug), "chunk": i, "text": text}))
    return points

print(len(chunk_points(50)), "points for 50-word chunks; first payload:",
      {k: v for k, v in chunk_points(50)[0].payload.items() if k != "text"})

95 points for 50-word chunks; first payload: {'slug': 'margherita-pizza', 'recipe_id': 'c1d62a2d-b942-5f7a-a84d-66729da747fb', 'chunk': 0}


Create the collection and load it. The count must equal the number of chunks built.

In [12]:
client = QdrantClient(":memory:")
VECTORS = models.VectorParams(size=384, distance=models.Distance.COSINE)
client.create_collection("chunks", vectors_config=VECTORS)
points = chunk_points(50)
client.upload_points("chunks", points)
print("chunks built:", len(points), "| points stored:", client.count("chunks").count)

chunks built: 95 | points stored: 95


### 6. Plain search returns chunks, not recipes
Ask the dough question with `query_points`, as in lesson 10:

In [13]:
q = "how long do I rest the dough?"
hits = client.query_points("chunks", query=embed_query(q), limit=6).points
for h in hits:
    print(f"{h.score:.3f}  {h.payload['slug']:<24} chunk {h.payload['chunk']}")

0.543  chocolate-chip-cookies   chunk 2
0.534  chocolate-chip-cookies   chunk 1
0.530  margherita-pizza         chunk 1
0.481  lasagne                  chunk 4
0.473  vegetable-samosas        chunk 1
0.450  margherita-pizza         chunk 2


Six hits, but only four recipes: cookies and pizza each fill two places. With long documents
this gets worse, as one recipe can take the whole page. Users want *recipes*, each with its
best passage.

### 7. `query_points_groups`: one result per recipe
`query_points_groups` searches the chunks and groups the hits by a payload field. `limit` is
the number of **groups** (recipes); `group_size` is the most hits kept per group. The field
must hold keyword or integer values. On a server, create a keyword payload index on it
(lesson 11), because grouping reads it for every candidate.

In [14]:
result = client.query_points_groups("chunks", query=embed_query(q), group_by="slug",
                                    limit=4, group_size=1)
for g in result.groups:
    best = g.hits[0]
    print(f"{best.score:.3f}  {g.id:<24} chunk {best.payload['chunk']}")
pizza = next(g for g in result.groups if g.id == "margherita-pizza")
print("\npizza's best passage ends: ...", " ".join(pizza.hits[0].payload["text"].split()[-14:]))

0.543  chocolate-chip-cookies   chunk 2
0.530  margherita-pizza         chunk 1
0.481  lasagne                  chunk 4
0.473  vegetable-samosas        chunk 1

pizza's best passage ends: ... dough in an oiled bowl, cover with a damp tea towel and leave it


Four groups, four different recipes, each with its best chunk. `g.id` is the `slug` value the
hits share, and each group's score is its best chunk's score. The pizza is found, but its best
passage is chunk 1, which ends at "leave it": the time, "1 to 2 hours", is in chunk 2. The
right recipe with half the answer. Keep that in mind for step 8 and step 11.

### 8. Fetch the parent with `with_lookup`
A chunk's payload should stay small, so recipe-level data (title, cuisine, minutes) lives once
in a `recipes` collection, keyed by `recipe_id(slug)`. Here its vector is the whole-method
embedding.

In [15]:
client.create_collection("recipes", vectors_config=VECTORS)
client.upload_points("recipes", [
    models.PointStruct(id=recipe_id(slug), vector=embed([docs[slug]])[0],
                       payload={"slug": slug, "title": recipes.title[slug],
                                "cuisine": recipes.cuisine[slug], "minutes": int(recipes.minutes[slug])})
    for slug in docs])
print("recipes stored:", client.count("recipes").count)

recipes stored: 24


Group by `recipe_id` and pass `with_lookup`: Qdrant uses each group's id as a point id in the
`recipes` collection and attaches that point as `g.lookup`. This time keep two chunks per recipe.

In [16]:
result = client.query_points_groups(
    "chunks", query=embed_query(q), group_by="recipe_id", limit=3, group_size=2,
    with_lookup=models.WithLookup(collection="recipes", with_payload=["title", "minutes"]))
for g in result.groups:
    print(f"{g.lookup.payload['title']} ({g.lookup.payload['minutes']} min):",
          [(h.payload["chunk"], round(h.score, 3)) for h in g.hits])

Chocolate chip cookies (25 min): [(2, 0.543), (1, 0.534)]
Margherita pizza (25 min): [(1, 0.53), (2, 0.45)]
Lasagne (90 min): [(4, 0.481), (3, 0.384)]


Each group now has its parent's title and minutes plus its two best chunks. The pizza's two
are chunks 1 and 2, which together hold the whole resting step. `WithLookup` chooses which of
the parent's payload fields come back, like `with_payload` (lesson 09). Always set it. Leaving
it out behaves differently by backend: a Qdrant server returns the parent's whole payload
(default `true`), while local mode returns `payload=None`.

### 9. Which chunk size retrieves best? Labels first
Lesson 16 taught the method: write an answer key, then measure. These labels are for questions
about the *steps*, graded per recipe (2 = this recipe's method answers it, 1 = a related
answer). None names a dish, the way you ask while cooking.

In [17]:
qrels = {
    QUERIES[0]: {"margherita-pizza": 2, "vegetable-samosas": 2, "apple-pie": 1, "chocolate-chip-cookies": 1},
    QUERIES[1]: {"chocolate-brownies": 2, "roast-chicken": 2, "lasagne": 1},
    QUERIES[2]: {"vegetable-samosas": 2, "falafel-wrap": 2},
    QUERIES[3]: {"roast-chicken": 2},
    QUERIES[4]: {"spaghetti-carbonara": 2, "vanilla-ice-cream": 2},
    QUERIES[5]: {"sushi-rolls": 2},
    QUERIES[6]: {"french-onion-soup": 2},
    QUERIES[7]: {"grilled-salmon": 2, "roast-chicken": 1},
    QUERIES[8]: {"strawberry-cheesecake": 2, "tiramisu": 2, "panna-cotta": 2},
    QUERIES[9]: {"panna-cotta": 2},
}
print(len(qrels), "queries labelled so far")

10 queries labelled so far


The second ten:

In [18]:
qrels |= {
    QUERIES[10]: {"hummus": 2, "falafel-wrap": 2},
    QUERIES[11]: {"margherita-pizza": 2, "thai-green-curry": 2},
    QUERIES[12]: {"lasagne": 2, "apple-pie": 2, "chocolate-brownies": 1, "roast-chicken": 1},
    QUERIES[13]: {"vanilla-ice-cream": 2, "lemon-sorbet": 2},
    QUERIES[14]: {"shrimp-pad-thai": 2},
    QUERIES[15]: {"ramen": 2, "beef-stew": 1},
    QUERIES[16]: {"apple-pie": 2, "margherita-pizza": 1},
    QUERIES[17]: {"pancakes": 2, "grilled-salmon": 1},
    QUERIES[18]: {"chocolate-chip-cookies": 2, "apple-pie": 2},
    QUERIES[19]: {"thai-green-curry": 2},
}
assert all(slug in docs for labels in qrels.values() for slug in labels)
with open("../data/eval_steps_20.json", "w") as f:
    json.dump({"model": MODEL, "qrels": qrels}, f, indent=1)
print(len(qrels), "queries,", sum(map(len, qrels.values())), "labelled pairs, saved to ../data/eval_steps_20.json")

20 queries, 40 labelled pairs, saved to ../data/eval_steps_20.json


Lesson 16's metrics, re-declared. They score a **run** of recipe slugs, so the run comes from
`query_points_groups` with `group_size=1`: one slug per recipe, in rank order. We also record
how many words the user must read: the length of the top recipe's best chunk.

In [19]:
def recall_at_k(ranked, labels, k):
    return sum(s in labels for s in ranked[:k]) / len(labels)

def reciprocal_rank(ranked, labels):
    return next((1 / rank for rank, s in enumerate(ranked, 1) if s in labels), 0.0)

def ndcg_at_k(ranked, labels, k):
    dcg = sum(labels.get(s, 0) / math.log2(r + 1) for r, s in enumerate(ranked[:k], 1))
    ideal = sum(g / math.log2(r + 1) for r, g in enumerate(sorted(labels.values(), reverse=True)[:k], 1))
    return dcg / ideal

print(recall_at_k(["a", "x", "b"], {"a": 2, "b": 1}, 3), reciprocal_rank(["x", "a"], {"a": 2}),
      round(ndcg_at_k(["a", "b"], {"a": 2, "b": 1}, 2), 3))

1.0 0.5 1.0


A sanity check on a toy run: it should print 1.0, 0.5, 1.0 (both relevant items found, the first
hit at rank 2, and the ideal order).

`evaluate_chunking` builds a fresh collection for one `(size, overlap)`, runs all 20 questions,
and returns the mean of each metric.

In [20]:
def evaluate_chunking(size, overlap=0, k=5):
    name = f"chunks_{size}_{overlap}"
    client.create_collection(name, vectors_config=VECTORS)
    points = chunk_points(size, overlap)
    client.upload_points(name, points)
    rows = []
    for q, labels in qrels.items():
        groups = client.query_points_groups(name, query=embed_query(q), group_by="slug",
                                            limit=k, group_size=1).groups
        ranked = [g.id for g in groups]
        rows.append({"MRR": reciprocal_rank(ranked, labels), "R@3": recall_at_k(ranked, labels, 3),
                     f"nDCG@{k}": ndcg_at_k(ranked, labels, k),
                     "words to read": len(groups[0].hits[0].payload["text"].split())})
    client.delete_collection(name)
    return {"chunks": len(points), **pd.DataFrame(rows).mean().to_dict()}

print({m: round(v, 3) for m, v in evaluate_chunking(50).items()})

{'chunks': 95, 'MRR': 0.975, 'R@3': 0.833, 'nDCG@5': 0.881, 'words to read': 44.65}


### 10. Measure: chunk size
Run every size with no overlap. `None` is the whole method as one chunk: the "no chunking"
baseline.

In [21]:
sizes = pd.DataFrame({("whole" if s is None else s): evaluate_chunking(s, 0)
                      for s, o in CONFIGS if o == 0}).T.rename_axis("chunk size")
print(sizes.round(3).to_string())

            chunks    MRR    R@3  nDCG@5  words to read
chunk size                                             
whole         24.0  0.842  0.792   0.780         155.70
200           31.0  0.860  0.742   0.812         119.15
100           55.0  0.925  0.858   0.847          67.15
50            95.0  0.975  0.833   0.881          44.65
25           178.0  0.938  0.817   0.896          24.65
12           353.0  0.879  0.783   0.826          12.00


Read the table top to bottom. The whole method as one vector is the weakest on MRR (0.842) and
nDCG@5 (0.780), and the user gets about 156 words to read. Cutting it up helps: 50-word chunks
reach MRR 0.975 and nDCG@5 0.881, 25-word chunks nDCG@5 0.896, and the passage shrinks to 45 or
25 words. At 12 words the scores turn down again (MRR 0.879, nDCG@5 0.826): a dozen words is
often half a sentence, with too little context to match a question.

Between 25 and 100 words the differences are a few hundredths. With 20 queries one query can
move a mean by up to 0.05 (lesson 16), so treat that range as a tie and choose inside it by what
the reader needs. Here about 50 words, two steps, is a good default.

### 11. Measure: overlap
Overlap protects facts that sit across a boundary. Count it directly: for each labelled
**answer** (the exact words that answer a question in a recipe), is it whole inside at least one
chunk? That needs no model, just text. We wrote exact answer spans for 20 of them, in 12 recipes;
the rest have no single quotable phrase.

In [22]:
answers = {
    "margherita-pizza": ["1 to 2 hours", "only after it comes out of the oven"],
    "vegetable-samosas": ["let it rest for 30 minutes", "160 to 170 C"],
    "chocolate-brownies": ["comes out with a few moist crumbs"],
    "roast-chicken": ["thermometer should read 75 C", "rest for 15 to 20 minutes"],
    "spaghetti-carbonara": ["off the heat so the eggs thicken"],
    "vanilla-ice-cream": ["Never let it boil", "every 30 minutes for 3 hours"],
    "strawberry-cheesecake": ["at least 6 hours, ideally overnight"],
    "lasagne": ["Resting lets the layers set"],
    "apple-pie": ["so the juices thicken", "helps the bottom crust cook through", "butter firms up"],
    "pancakes": ["Wait until bubbles appear all over the surface"],
    "falafel-wrap": ["overnight, at least 12 hours", "175 C"],
    "thai-green-curry": ["until it smells fragrant", "at the very end"],
}
assert all(a in docs[s] for s, spans in answers.items() for a in spans)
print(sum(map(len, answers.values())), "answers in", len(answers), "recipes")

20 answers in 12 recipes


`split_answers` lists the answers that no chunk contains whole.

In [23]:
def split_answers(size, overlap=0):
    return [(s, a) for s, spans in answers.items() for a in spans
            if not any(a in c for c in chunks_for(s, size, overlap))]

for size, overlap in [(50, 0), (50, 10), (25, 0), (25, 10), (12, 0)]:
    cut = split_answers(size, overlap)
    print(f"size {size:>2}, overlap {overlap:>2}: {len(cut)} answers cut", cut[:2])

size 50, overlap  0: 2 answers cut [('spaghetti-carbonara', 'off the heat so the eggs thicken'), ('thai-green-curry', 'until it smells fragrant')]
size 50, overlap 10: 0 answers cut []
size 25, overlap  0: 2 answers cut [('spaghetti-carbonara', 'off the heat so the eggs thicken'), ('thai-green-curry', 'until it smells fragrant')]
size 25, overlap 10: 0 answers cut []
size 12, overlap  0: 7 answers cut [('margherita-pizza', '1 to 2 hours'), ('vegetable-samosas', '160 to 170 C')]


With 50 or 25 words and no overlap, two answers fall across a boundary: carbonara's "off the
heat so the eggs thicken" and the curry's "until it smells fragrant". No chunk holds them whole.
An overlap of 10 words brings that to 0. With 12-word chunks, 7 of the 20 answers are cut.

This test is generous: it asks whether *some* chunk holds the answer, not whether the best
matching one does. The pizza passes it ("1 to 2 hours" is whole in chunk 2), yet in step 7 its
best chunk was chunk 1.

Now the retrieval side: the same evaluation with overlap.

In [24]:
overlaps = pd.DataFrame({f"{s} / {o}": evaluate_chunking(s, o)
                         for s, o in [(50, 0), (50, 10), (50, 25), (25, 0), (25, 10)]}).T
print(overlaps.rename_axis("size / overlap").round(3).to_string())

                chunks    MRR    R@3  nDCG@5  words to read
size / overlap                                             
50 / 0            95.0  0.975  0.833   0.881          44.65
50 / 10          110.0  0.892  0.871   0.885          46.50
50 / 25          154.0  0.925  0.817   0.888          47.20
25 / 0           178.0  0.938  0.817   0.896          24.65
25 / 10          273.0  0.892  0.783   0.819          24.60


Overlap does not improve the ranking here. For 50-word chunks, 10 words of overlap lowers MRR
(0.975 to 0.892) and raises R@3 (0.833 to 0.871); nDCG@5 hardly moves (0.881 to 0.885). For
25-word chunks, 10 words of overlap is lower on all three. Those moves are one or two queries
each (lesson 16: one query moves a mean by up to 0.05), too few to call overlap better or worse.
What overlap certainly costs is points: 95, then 110, then
154 for the same 50-word chunks.

So overlap buys complete passages (the previous table), not better ranking. A small overlap
(10-20% of the chunk) is cheap insurance; more mostly adds points. Measure on your own data
before going further: these numbers are for 24 recipes and MiniLM.

## What just happened
- The 24 methods are 95-324 words; 5 are longer than MiniLM's 256 tokens, and removing the pizza's last step left its vector unchanged (cosine 1.0): the model never read it.
- `chunk_words` cut each method into windows of words; at 50 words the pizza became 7 chunks, each stored as its own point with its parent's `slug` and `recipe_id` (95 points).
- Plain `query_points` returned 6 chunks from only 4 recipes; `query_points_groups(group_by="slug")` returned one group per recipe, and `with_lookup` attached each recipe's title and minutes.
- On 20 labelled step questions, 25-100-word chunks beat whole methods (MRR 0.842 → 0.975 at 50 words; nDCG@5 0.780 → 0.896 at 25 words) and cut reading from 156 to 25-67 words; 12-word chunks fell back.
- Overlap of 10 words reduced answers cut by a boundary from 2 to 0 without improving ranking, and added points (95 to 110).

## Common mistakes

**1. Overlap as big as the chunk.** The window moves forward by `size - overlap` words. When they
are equal, it never moves:

In [25]:
try:
    chunk_words(docs["pancakes"], size=50, overlap=50)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: range() arg 3 must not be zero


It prints `ValueError: range() arg 3 must not be zero`. With overlap *larger* than size, the step
is negative and `range` returns nothing: an empty list and no error, which is worse. Fix: keep
overlap well below size (10-20% is a common start) and check it in the chunker:

In [26]:
def chunk_words(text, size, overlap=0):
    if not 0 <= overlap < size:
        raise ValueError(f"overlap must be in [0, size), got size={size}, overlap={overlap}")
    words = text.split()
    return [" ".join(words[i:i + size]) for i in range(0, max(len(words) - overlap, 1), size - overlap)]

print(len(chunk_words(docs["pancakes"], size=50, overlap=10)), "chunks")

5 chunks


**2. Giving every chunk its parent's id.** `recipe_id(slug)` is the *recipe's* id. Use it for
each chunk, and each upsert replaces the previous chunk of the same recipe (lesson 09): only the
last chunk survives, with no error.

In [27]:
try:
    client.create_collection("chunks_bad", vectors_config=VECTORS)
    bad = [models.PointStruct(id=recipe_id(p.payload["slug"]), vector=p.vector, payload=p.payload)
           for p in chunk_points(50)]
    client.upload_points("chunks_bad", bad)
    stored = client.count("chunks_bad").count
    assert stored == len(bad), f"built {len(bad)} chunks but stored {stored}: chunk ids collide"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

AssertionError: built 95 chunks but stored 24: chunk ids collide


It prints `AssertionError: built 95 chunks but stored 24: chunk ids collide`. The first chunks
of every recipe, including the pizza's resting step, are gone. Fix: one id per chunk
(`uuid5` of `slug#index`, as in `chunk_points`), the parent's id only in the payload, and always
compare the count with the number built.

In [28]:
client.delete_collection("chunks_bad")
print("points in 'chunks':", client.count("chunks").count, "(one per chunk)")

points in 'chunks': 95 (one per chunk)


## Try it
1. **One step per chunk.** The methods are already split into steps. Build a collection where
   each step is a chunk (every step is in the cache), evaluate it with the same 20 questions,
   and compare it with the 50-word windows. Does respecting sentence boundaries help?
2. **Show the answer, not just the recipe.** For "how long does it need to set in the fridge?",
   print the top 3 recipes with their title (via `with_lookup`) and their two best passages.
   Then try `group_size=3` with `limit=1`: what do you get?

In [29]:
# your code here

Clean up: close the local client.

In [30]:
client.close()
print("client closed")

client closed


## Key terms
- **chunk** — a piece of a longer document that gets its own vector and point; it stores its parent's key and its own text.
- **chunking** — splitting long documents into chunks before embedding, one focused passage per vector.
- **chunk size** — how much text goes in one chunk (here, words); too big blurs the answer, too small loses context.
- **overlap** — words a chunk repeats from the end of the previous one, so a fact cut by a boundary is still whole in one chunk.
- **parent document** — the whole item a chunk came from (here, a recipe); results are usually shown per parent.
- **max sequence length** — the most tokens an embedding model reads (256 for MiniLM); the rest of a longer text is silently ignored.
- **query_points_groups** — Qdrant search that groups hits by a payload field: `limit` groups, up to `group_size` hits each.
- **with_lookup** — attaches to each group the point with the group's id from another collection (the parent).